# 02 - Materialize matched GTFS data into `ml`

The only notebook that touches `silver.gtfs_*`. Resolves the correct GTFS
feed/route/direction for every `(route_id, trip_date)` pair once, copies
the matched shapes and scheduled-trip durations into `ml`, and stores
which feed/route/shape was used for traceability. Every later notebook
(and any rebuild of `03_trip_metrics`) reads only from the three tables
built here plus `ml.trip_validity_trips` — `silver` is never touched
again after this notebook runs.

Key facts established earlier by direct exploration, that this notebook
relies on:

- GTFS `route_id` is always 4-digit zero-padded (`"0031"`), unrelated to
  our own `route_id`'s minimum-3-digit padding — the correct join key is
  `route_short_name` (the plain unpadded number, 1:1 with `route_id`),
  normalized by stripping leading zeros from our own `route_id`.
- `gtfs_trips.direction_id` is 100% NULL in this data. Direction instead
  lives as a literal suffix on `shape_id` itself: `shape0060-I` /
  `shape0211-V` (confirmed: every shape_id in the Nov-2023 feed ends in
  exactly `-I` or `-V`).
- `shape_dist_traveled` is unpopulated everywhere (both
  `gtfs_shapes` and `gtfs_stop_times`) — shape length has to be computed
  from the ordered points, not looked up.
- `stop_times.arrival_time`/`departure_time` are text and can exceed
  `24:00:00` for late-night service (standard GTFS convention) — parsed
  manually into seconds-from-midnight, not cast to a Postgres `time`.
- 30 of the 352 routes appearing in November 2023 trips never appear in
  *any* GTFS feed, ever — those permanently resolve to NULL.
- A minority of routes (13 in the Nov-2023 feed) split direction into two
  entirely separate `route_id`/`route_short_name` pairs instead of one
  `route_id` with two shapes. This notebook does not attempt to
  auto-link those sibling route numbers — it would be a heuristic guess,
  not a real match — so those routes will show `has_both_directions =
  false` even though the physical route has two directions.

In [1]:
import os
from pathlib import Path

import psycopg
from psycopg import sql

In [2]:
_root = Path.cwd()
while not (_root / "pyproject.toml").exists():
    _root = _root.parent
os.chdir(_root)
os.environ.setdefault("RAW_DATA_ROOT", str(_root))

'/home/victor/repos/opa-database'

In [3]:
from opa_database.config import settings

conn = psycopg.connect(settings.db_dsn)
conn.execute("CREATE SCHEMA IF NOT EXISTS ml;")
conn.commit()
print("connected")

connected


## Stage 1 - `ml.trip_validity_route_gtfs_match`

One row per distinct `(route_id, trip_date)` pair actually present in
`ml.trip_validity_trips` (not per trip — a tiny lookup table, resolved
once). For each pair:

1. Normalize our `route_id` by stripping leading zeros, and find every
   `silver.gtfs_routes` row across *all* feeds whose `route_short_name`
   matches.
2. Among those candidates, pick the one with `feed_version_date <=
   trip_date` closest to `trip_date`; if none exists, the closest
   `feed_version_date > trip_date`; if no candidates at all, NULL
   (the route never appears in any feed).
3. From that resolved `(feed_version_date, gtfs_route_id)`, look up
   which of the `-I`/`-V` shape suffixes actually have trips.

In [4]:
conn.execute("""
    DROP TABLE IF EXISTS ml.trip_validity_route_gtfs_match CASCADE;

    CREATE TABLE ml.trip_validity_route_gtfs_match (
        route_id                text NOT NULL,
        trip_date                date NOT NULL,
        gtfs_feed_version_date   date,
        gtfs_route_id            text,
        gtfs_route_short_name    text,
        gtfs_shape_id_i          text,
        gtfs_shape_id_v          text,
        gtfs_route_has_both_directions boolean,
        PRIMARY KEY (route_id, trip_date)
    );
""")

conn.execute("""
    INSERT INTO ml.trip_validity_route_gtfs_match (
        route_id, trip_date, gtfs_feed_version_date,
        gtfs_route_id, gtfs_route_short_name
    )
    SELECT
        rd.route_id,
        rd.trip_date,
        resolved.feed_version_date,
        resolved.route_id,
        resolved.route_short_name
    FROM (
        SELECT DISTINCT route_id, trip_date FROM ml.trip_validity_trips
    ) rd
    LEFT JOIN LATERAL (
        SELECT r.feed_version_date, r.route_id, r.route_short_name
        FROM silver.gtfs_routes r
        WHERE r.route_short_name = ltrim(rd.route_id, '0')
        ORDER BY
            CASE WHEN r.feed_version_date <= rd.trip_date THEN 0 ELSE 1 END,
            CASE WHEN r.feed_version_date <= rd.trip_date
                 THEN rd.trip_date - r.feed_version_date
                 ELSE r.feed_version_date - rd.trip_date
            END ASC
        LIMIT 1
    ) resolved ON true;
""")

conn.execute("""
    UPDATE ml.trip_validity_route_gtfs_match m
    SET gtfs_shape_id_i = shapes.shape_id_i,
        gtfs_shape_id_v = shapes.shape_id_v,
        gtfs_route_has_both_directions = (
            shapes.shape_id_i IS NOT NULL AND shapes.shape_id_v IS NOT NULL
        )
    FROM (
        SELECT
            feed_version_date,
            route_id,
            max(shape_id) FILTER (WHERE shape_id LIKE '%-I') AS shape_id_i,
            max(shape_id) FILTER (WHERE shape_id LIKE '%-V') AS shape_id_v
        FROM silver.gtfs_trips
        WHERE shape_id IS NOT NULL
        GROUP BY feed_version_date, route_id
    ) shapes
    WHERE shapes.feed_version_date = m.gtfs_feed_version_date
      AND shapes.route_id = m.gtfs_route_id;
""")

conn.execute("""
    CREATE INDEX trip_validity_route_gtfs_match_feed_idx
        ON ml.trip_validity_route_gtfs_match (gtfs_feed_version_date, gtfs_route_id);
""")
conn.execute("ANALYZE ml.trip_validity_route_gtfs_match;")
conn.commit()

with conn.cursor() as cur:
    cur.execute("""
        SELECT
            count(*) AS pairs,
            count(DISTINCT route_id) AS routes,
            count(DISTINCT route_id)
                FILTER (WHERE gtfs_feed_version_date IS NULL) AS orphan_routes,
            count(*)
                FILTER (WHERE gtfs_route_has_both_directions) AS pairs_both_directions
        FROM ml.trip_validity_route_gtfs_match;
    """)
    print(cur.fetchone())

(9262, 352, 30, 8348)


## Stage 2 - `ml.trip_validity_route_shapes`

One row per distinct `(feed_version_date, shape_id)` actually referenced
by Stage 1 (not one row per raw shape point) — the geometry is built
once here from `silver.gtfs_shapes`'s ordered points via
`ST_MakeLine(geom ORDER BY shape_pt_sequence)`, so notebook 03 never
touches raw shape points.

Stores both the original `EPSG:4326` geometry (for reference/mapping)
and a version pre-projected to `EPSG:31984` (SIRGAS2000 / UTM 24S, the
standard metric CRS for Ceará) — `ST_FrechetDistance`/
`ST_HausdorffDistance` have no `geography` overload, so this avoids
re-projecting on every trip comparison in notebook 03.

In [5]:
conn.execute("""
    DROP TABLE IF EXISTS ml.trip_validity_route_shapes CASCADE;

    CREATE TABLE ml.trip_validity_route_shapes (
        feed_version_date  date NOT NULL,
        shape_id           text NOT NULL,
        route_short_name   text,
        direction          text NOT NULL CHECK (direction IN ('I', 'V')),
        shape_geom         geometry(LineString, 4326),
        shape_geom_metric  geometry(LineString, 31984),
        shape_length_meters double precision,
        PRIMARY KEY (feed_version_date, shape_id)
    );
""")

conn.execute("""
    WITH needed_shapes AS (
        SELECT DISTINCT gtfs_feed_version_date AS feed_version_date,
               gtfs_shape_id_i AS shape_id,
               gtfs_route_short_name AS route_short_name, 'I' AS direction
        FROM ml.trip_validity_route_gtfs_match WHERE gtfs_shape_id_i IS NOT NULL
        UNION
        SELECT DISTINCT gtfs_feed_version_date, gtfs_shape_id_v,
               gtfs_route_short_name, 'V'
        FROM ml.trip_validity_route_gtfs_match WHERE gtfs_shape_id_v IS NOT NULL
    )
    INSERT INTO ml.trip_validity_route_shapes (
        feed_version_date, shape_id, route_short_name, direction,
        shape_geom, shape_geom_metric, shape_length_meters
    )
    SELECT
        s.feed_version_date,
        s.shape_id,
        s.route_short_name,
        s.direction,
        line.geom,
        ST_Transform(line.geom, 31984),
        ST_Length(line.geom::geography)
    FROM needed_shapes s
    JOIN LATERAL (
        SELECT ST_MakeLine(geom ORDER BY shape_pt_sequence) AS geom
        FROM silver.gtfs_shapes
        WHERE feed_version_date = s.feed_version_date AND shape_id = s.shape_id
    ) line ON line.geom IS NOT NULL;
""")

conn.execute("""
    CREATE INDEX trip_validity_route_shapes_geom_idx
        ON ml.trip_validity_route_shapes USING GIST (shape_geom);
""")
conn.execute("ANALYZE ml.trip_validity_route_shapes;")
conn.commit()

with conn.cursor() as cur:
    cur.execute("""
        SELECT count(*),
               count(*) FILTER (WHERE direction = 'I'),
               count(*) FILTER (WHERE direction = 'V')
        FROM ml.trip_validity_route_shapes;
    """)
    print("shapes total / I / V:", cur.fetchone())

shapes total / I / V: (1864, 931, 933)


## Stage 3 - `ml.trip_validity_route_schedule`

One row per scheduled GTFS trip (a *scheduled service run*, not one of
our AFC trips — different concept, same word, deliberately named
`gtfs_trip_id` here to keep the two unambiguous) on a matched route.
`scheduled_duration_seconds` is `MAX - MIN` of each stop's
`COALESCE(departure_time, arrival_time)`, parsed manually since GTFS
times are text and can exceed `24:00:00`.

`ml.parse_gtfs_time` is a small helper function for that parsing, kept
in the `ml` schema since it's reusable and specific to GTFS's time
format.

In [6]:
conn.execute("""
    CREATE OR REPLACE FUNCTION ml.parse_gtfs_time(t text) RETURNS integer AS $$
        SELECT split_part(t, ':', 1)::int * 3600
             + split_part(t, ':', 2)::int * 60
             + split_part(t, ':', 3)::int;
    $$ LANGUAGE sql IMMUTABLE STRICT;
""")

conn.execute("""
    DROP TABLE IF EXISTS ml.trip_validity_route_schedule CASCADE;

    CREATE TABLE ml.trip_validity_route_schedule (
        feed_version_date            date NOT NULL,
        gtfs_route_id                text NOT NULL,
        direction                    text NOT NULL CHECK (direction IN ('I', 'V')),
        gtfs_trip_id                 text NOT NULL,
        scheduled_duration_seconds   integer,
        start_hour                   smallint,
        PRIMARY KEY (feed_version_date, gtfs_trip_id)
    );
""")

conn.execute("""
    WITH needed_trips AS (
        SELECT t.feed_version_date, t.trip_id AS gtfs_trip_id,
               t.route_id AS gtfs_route_id, right(t.shape_id, 1) AS direction
        FROM silver.gtfs_trips t
        JOIN (
            SELECT DISTINCT gtfs_feed_version_date AS feed_version_date, gtfs_route_id
            FROM ml.trip_validity_route_gtfs_match
            WHERE gtfs_route_id IS NOT NULL
        ) nr
          ON nr.feed_version_date = t.feed_version_date
         AND nr.gtfs_route_id = t.route_id
        WHERE t.shape_id LIKE '%-I' OR t.shape_id LIKE '%-V'
    ),
    trip_times AS (
        SELECT
            nt.feed_version_date, nt.gtfs_route_id, nt.direction, nt.gtfs_trip_id,
            min(ml.parse_gtfs_time(coalesce(st.departure_time, st.arrival_time)))
                AS start_sec,
            max(ml.parse_gtfs_time(coalesce(st.departure_time, st.arrival_time)))
                AS end_sec
        FROM needed_trips nt
        JOIN silver.gtfs_stop_times st
          ON st.feed_version_date = nt.feed_version_date
         AND st.trip_id = nt.gtfs_trip_id
        GROUP BY nt.feed_version_date, nt.gtfs_route_id, nt.direction, nt.gtfs_trip_id
    )
    INSERT INTO ml.trip_validity_route_schedule (
        feed_version_date, gtfs_route_id, direction, gtfs_trip_id,
        scheduled_duration_seconds, start_hour
    )
    SELECT
        feed_version_date, gtfs_route_id, direction, gtfs_trip_id,
        end_sec - start_sec,
        (start_sec / 3600) % 24
    FROM trip_times
    WHERE start_sec IS NOT NULL AND end_sec IS NOT NULL;
""")

conn.execute("""
    CREATE INDEX trip_validity_route_schedule_lookup_idx
        ON ml.trip_validity_route_schedule
            (feed_version_date, gtfs_route_id, direction);
    CREATE INDEX trip_validity_route_schedule_hour_idx
        ON ml.trip_validity_route_schedule
            (feed_version_date, gtfs_route_id, direction, start_hour);
""")
conn.execute("ANALYZE ml.trip_validity_route_schedule;")
conn.commit()

with conn.cursor() as cur:
    cur.execute("""
        SELECT count(*), count(*) FILTER (WHERE scheduled_duration_seconds < 0)
        FROM ml.trip_validity_route_schedule;
    """)
    print("scheduled trips / negative durations:", cur.fetchone())

scheduled trips / negative durations: (186540, 0)


In [7]:
def comment_on_column(cur: psycopg.Cursor, table: str, col: str, text: str) -> None:
    """Apply a COMMENT ON COLUMN for one column via safe SQL composition."""
    cur.execute(
        sql.SQL("COMMENT ON COLUMN ml.{}.{} IS {};").format(
            sql.Identifier(table), sql.Identifier(col), sql.Literal(text)
        )
    )


MATCH_COMMENTS = {
    "route_id": ("ml.trip_validity_trips.route_id (our own zero-padded route number)."),
    "trip_date": "ml.trip_validity_trips.trip_date.",
    "gtfs_feed_version_date": (
        "Resolved GTFS feed: closest feed_version_date <= trip_date among "
        "feeds containing this route (by route_short_name); else closest "
        "feed_version_date > trip_date; else NULL if the route never "
        "appears in any feed."
    ),
    "gtfs_route_id": (
        "The matched silver.gtfs_routes.route_id (4-digit padded, NOT the "
        "same padding scheme as our own route_id) in the resolved feed. "
        "NULL if unresolved."
    ),
    "gtfs_route_short_name": (
        "The matched silver.gtfs_routes.route_short_name (unpadded) - the "
        "actual join key used. Stored for traceability."
    ),
    "gtfs_shape_id_i": (
        "silver.gtfs_trips.shape_id ending in '-I' for this route+feed, if any."
    ),
    "gtfs_shape_id_v": (
        "silver.gtfs_trips.shape_id ending in '-V' for this route+feed, if any."
    ),
    "gtfs_route_has_both_directions": (
        "True iff both an -I and a -V shape exist for this route in the "
        "resolved feed. Note: ~13 routes (in the Nov-2023 feed) split "
        "direction into two separate route_ids instead of one route_id "
        "with two shapes - those show false here even though the "
        "physical route has two directions."
    ),
}
SHAPES_COMMENTS = {
    "feed_version_date": "silver.gtfs_shapes/gtfs_trips.feed_version_date, as-is.",
    "shape_id": (
        "silver.gtfs_shapes.shape_id, as-is. Direction is encoded as a "
        "literal '-I'/'-V' suffix by the source data."
    ),
    "route_short_name": (
        "Carried over from trip_validity_route_gtfs_match for traceability."
    ),
    "direction": "'I' or 'V', parsed from the shape_id suffix.",
    "shape_geom": (
        "ST_MakeLine(geom ORDER BY shape_pt_sequence) over "
        "silver.gtfs_shapes points for this shape_id. SRID 4326."
    ),
    "shape_geom_metric": (
        "shape_geom transformed to EPSG:31984 (SIRGAS2000 / UTM 24S), for "
        "planar distance functions with no geography overload "
        "(Frechet/Hausdorff)."
    ),
    "shape_length_meters": (
        "ST_Length(shape_geom::geography) - the route's total length in this direction."
    ),
}
SCHEDULE_COMMENTS = {
    "feed_version_date": "silver.gtfs_trips.feed_version_date, as-is.",
    "gtfs_route_id": (
        "silver.gtfs_trips.route_id, as-is (the GTFS trip's own route, "
        "not our route_id)."
    ),
    "direction": "'I' or 'V', parsed from silver.gtfs_trips.shape_id suffix.",
    "gtfs_trip_id": (
        "silver.gtfs_trips.trip_id, as-is. NOTE: a GTFS "
        "scheduled-service-run id, unrelated to "
        "ml.trip_validity_trips.trip_id (our AFC fare-tap trip id) - "
        "different concept, same word in the GTFS spec."
    ),
    "scheduled_duration_seconds": (
        "MAX - MIN of ml.parse_gtfs_time(COALESCE(departure_time, "
        "arrival_time)) across this scheduled trip's "
        "silver.gtfs_stop_times rows."
    ),
    "start_hour": (
        "Hour bucket (0-23) of this scheduled trip's first stop time, "
        "computed as (start_seconds / 3600) % 24 so GTFS's >24:00:00 "
        "late-night convention still buckets to a comparable hour-of-day."
    ),
}

with conn.cursor() as cur:
    for col, text in MATCH_COMMENTS.items():
        comment_on_column(cur, "trip_validity_route_gtfs_match", col, text)
    for col, text in SHAPES_COMMENTS.items():
        comment_on_column(cur, "trip_validity_route_shapes", col, text)
    for col, text in SCHEDULE_COMMENTS.items():
        comment_on_column(cur, "trip_validity_route_schedule", col, text)

conn.execute(
    sql.SQL("COMMENT ON FUNCTION ml.parse_gtfs_time(text) IS {};").format(
        sql.Literal(
            "Parses a GTFS HH:MM:SS time string (H may exceed 24 for "
            "late-night service) into seconds-from-midnight. STRICT: NULL "
            "in, NULL out."
        )
    )
)
TABLE_COMMENTS = [
    (
        "trip_validity_route_gtfs_match",
        "Trip Validity model: resolved GTFS feed/route/shape for every "
        "(route_id, trip_date) pair in trip_validity_trips. See "
        "ml/trip_validity_model/notebooks/02_gtfs_materialize.ipynb.",
    ),
    (
        "trip_validity_route_shapes",
        "Trip Validity model: built shape geometries + lengths for every "
        "shape referenced by trip_validity_route_gtfs_match. See "
        "ml/trip_validity_model/notebooks/02_gtfs_materialize.ipynb.",
    ),
    (
        "trip_validity_route_schedule",
        "Trip Validity model: parsed scheduled-trip durations for every "
        "matched route in trip_validity_route_gtfs_match. See "
        "ml/trip_validity_model/notebooks/02_gtfs_materialize.ipynb.",
    ),
]
for table, text in TABLE_COMMENTS:
    conn.execute(
        sql.SQL("COMMENT ON TABLE ml.{} IS {};").format(
            sql.Identifier(table), sql.Literal(text)
        )
    )

conn.commit()
print("comments applied")

comments applied


## Verification

In [8]:
import polars as pl

EXPECTED_ROUTE_COUNT = 352  # distinct route_ids in November 2023 trips

with conn.cursor() as cur:
    cur.execute("""
        SELECT
            (SELECT count(*) FROM ml.trip_validity_route_gtfs_match) AS match_pairs,
            (SELECT count(DISTINCT route_id) FROM ml.trip_validity_route_gtfs_match)
                AS distinct_routes,
            (SELECT count(DISTINCT route_id) FROM ml.trip_validity_route_gtfs_match
                WHERE gtfs_feed_version_date IS NULL) AS orphan_routes,
            (SELECT count(*) FROM ml.trip_validity_route_shapes) AS shapes,
            (SELECT count(*) FROM ml.trip_validity_route_schedule) AS scheduled_trips,
            (SELECT count(*) FROM ml.trip_validity_route_schedule
                WHERE scheduled_duration_seconds < 0) AS negative_scheduled;
    """)
    cols = [d.name for d in cur.description]
    row = cur.fetchone()

summary = pl.DataFrame([dict(zip(cols, row, strict=True))])
print(summary)

if summary["negative_scheduled"][0] != 0:
    msg = "found negative scheduled durations - GTFS time parsing bug"
    raise AssertionError(msg)
if summary["distinct_routes"][0] != EXPECTED_ROUTE_COUNT:
    msg = f"expected {EXPECTED_ROUTE_COUNT} distinct routes from November 2023 trips"
    raise AssertionError(msg)
print(
    f"OK: {summary['orphan_routes'][0]} of {summary['distinct_routes'][0]} routes "
    "have no GTFS match in any feed (expected 30)"
)

shape: (1, 6)
┌─────────────┬─────────────────┬───────────────┬────────┬─────────────────┬────────────────────┐
│ match_pairs ┆ distinct_routes ┆ orphan_routes ┆ shapes ┆ scheduled_trips ┆ negative_scheduled │
│ ---         ┆ ---             ┆ ---           ┆ ---    ┆ ---             ┆ ---                │
│ i64         ┆ i64             ┆ i64           ┆ i64    ┆ i64             ┆ i64                │
╞═════════════╪═════════════════╪═══════════════╪════════╪═════════════════╪════════════════════╡
│ 9262        ┆ 352             ┆ 30            ┆ 1864   ┆ 186540          ┆ 0                  │
└─────────────┴─────────────────┴───────────────┴────────┴─────────────────┴────────────────────┘
OK: 30 of 352 routes have no GTFS match in any feed (expected 30)
